In [5]:
# ============================================================
# SVM (SVC) – Klassificeringsexperiment enligt instruktion
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [6]:
# ------------------------------------------------------------
# 1. Läs in dataset
# ------------------------------------------------------------
df = pd.read_csv("Pokemon.csv")

# Ta bort Type 2 och Legendary om de finns
df = df.drop(columns=["Type 2", "Legendary"], errors="ignore")

# Target = Type 1
y_raw = df["Type 1"]
features = ['HP','Attack','Defense','Sp. Atk','Sp. Def','Speed','Generation']
X_raw = df[features]

# Label-encode target (multiklass)
le = LabelEncoder()
y = le.fit_transform(y_raw)
class_labels = le.classes_

print(f"Dataset med {df.shape[0]} rader och {len(class_labels)} klasser.")

Dataset med 800 rader och 18 klasser.


In [ ]:
# ------------------------------------------------------------ 
# 2. Experimentinställningar 
# # ------------------------------------------------------------ 
scalings = ["original", "minmax"]
splits = { "75_25": 0.75, "60_40": 0.60 }
kernels = ["linear", "rbf"]
results = []

In [7]:
# ------------------------------------------------------------
# 3. Hjälpfunktion för confusion matrix-heatmap
# ------------------------------------------------------------
def plot_cm(cm, title):
    plt.figure(figsize=(10,7))
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d",
                xticklabels=class_labels, yticklabels=class_labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [8]:
# ------------------------------------------------------------
# 4. Kör alla kombinationer → 8 körningar
# ------------------------------------------------------------
for scaling in scalings:

    # Skala eller inte
    if scaling == "minmax":
        scaler = MinMaxScaler()
        X = scaler.fit_transform(X_raw)
    else:
        X = X_raw.values

    for split_name, train_size in splits.items():
        for kernel in kernels:

            # Train/test split
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, train_size=train_size, stratify=y, random_state=42
            )

            # SVM-modell
            clf = SVC(kernel=kernel)
            clf.fit(X_train, y_train)

            # Prediktion
            y_pred = clf.predict(X_test)

            # Metriker
            cm = confusion_matrix(y_test, y_pred)
            acc = accuracy_score(y_test, y_pred)

            # Kort kommentar
            comment = f"Kernel={kernel}, scaling={scaling}, split={split_name}."

            results.append({
                "scaling": scaling,
                "split": split_name,
                "kernel": kernel,
                "accuracy": acc,
                "confusion_matrix": cm,
                "comment": comment
            })

NameError: name 'scalings' is not defined

In [ ]:
# ------------------------------------------------------------
# 5. Sammanställ resultattabell
# ------------------------------------------------------------
summary = pd.DataFrame([{
    "scaling": r["scaling"],
    "split": r["split"],
    "kernel": r["kernel"],
    "accuracy": round(r["accuracy"], 4)
} for r in results])

summary_sorted = summary.sort_values(by="accuracy", ascending=False).reset_index(drop=True)

print("=== Resultattabell – alla 8 körningar ===")
display(summary_sorted)

In [ ]:
# ------------------------------------------------------------
# 6. Presentera Top 3
# ------------------------------------------------------------
top3 = summary_sorted.head(3)

print("\n=== TOPP 3 (baserat på accuracy) ===")
display(top3)

for i, row in top3.iterrows():
    match = next(r for r in results if 
                 r["scaling"] == row["scaling"] and
                 r["split"] == row["split"] and
                 r["kernel"] == row["kernel"])
    
    print(f"\n--- Rank {i+1} ---")
    print(f"Scaling: {match['scaling']}, Split: {match['split']}, Kernel: {match['kernel']}")
    print(f"Accuracy: {round(match['accuracy'], 4)}")
    display(pd.DataFrame(match["confusion_matrix"],
                         index=class_labels, columns=class_labels))
    
    plot_cm(match["confusion_matrix"],
            title=f"Confusion Matrix – Rank {i+1}")

In [ ]:
# ------------------------------------------------------------
# 7. Reflektion / slutsatser
# ------------------------------------------------------------
print("\n=== Reflektion ===")
print("""
1) Normaliserat vs icke normaliserat data
   SVM (särskilt RBF-kernel) är mycket känslig för skalor.
   Normaliserat data brukar ge betydligt bättre resultat.
   Linear-kernel påverkas mindre, men tjänar fortfarande på normalisering.

2) Effekt av kernel-funktioner
   - Linear fungerar bäst om klasserna har ungefär linjära gränser.
   - RBF brukar prestera bättre i komplexa, icke-linjära dataset.
   I Pokémon-stats är typgränserna inte linjära → RBF får ofta högre accuracy.

3) Effekt av olika train/test-fördelningar
   75/25 ger större testset → stabilare accuracy-uppskattning.
   60/40 ger mer träningsdata → ibland högre accuracy, men mer varians.

4) Allmänna observationer om SVM i detta experiment
   - Prestandan begränsas naturligt av att stats INTE korrelerar starkt
     med Pokémon-typen → accuracy blir generellt låg (ofta 0.2–0.35).
   - Normalisering är avgörande för att RBF-kernel ska fungera bra.
   - SVM hanterar multiklass genom one-vs-rest, vilket fungerar men kräver
     bra separationsstruktur i data, något Pokémon-stats inte riktigt har.
""")